In [2]:
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
import mlflow
import mlflow.xgboost
from mlflow.tracking import MlflowClient
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore", category=FutureWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

os.environ["LOGNAME"]  = "Yulia"
os.environ["USER"]     = "Yulia"
os.environ["USERNAME"] = "Yulia"

# Поиск корня проекта по маркеру PROJECT_CONTEXT.md
def find_project_root(marker="PROJECT_CONTEXT.md"):
    current = Path.cwd().resolve()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Не найден корень проекта (нет файла {marker})")

project_root = find_project_root()
print(f"Корень проекта: {project_root}")

data_path = project_root / "data" / "processed" / "training_set_v1.parquet"
print(f"Путь к данным : {data_path}")
print(f"Файл существует: {data_path.exists()}")

df = pd.read_parquet(data_path)
print(f"\nДатасет загружен: {df.shape}")
print(f"Период: {df['incident_hour'].min()} .. {df['incident_hour'].max()}")

Корень проекта: C:\metro-bus-ml
Путь к данным : C:\metro-bus-ml\data\processed\training_set_v1.parquet
Файл существует: True

Датасет загружен: (1394946, 14)
Период: 2024-10-01 00:00:00 .. 2024-12-31 22:00:00


In [3]:
# Feature engineering — 10 новых колонок
df["hour"]          = df["incident_hour"].dt.hour
df["day_of_week"]   = df["incident_hour"].dt.dayofweek
df["is_weekend"]    = (df["day_of_week"] >= 5).astype(int)
df["month"]         = df["incident_hour"].dt.month
df["day_of_month"]  = df["incident_hour"].dt.day

df["time_of_day"] = pd.cut(
    df["hour"],
    bins=[-1, 6, 10, 16, 20, 24],
    labels=["night", "morning_rush", "midday", "evening_rush", "evening"],
)

df["status_main"] = df["status_label"].str.split().str[0].fillna("unknown")
df["is_express"]  = df["bus_route"].str.contains(r"\+$|X$", regex=True, na=False).astype(int)

# Принадлежность маршрута затронутой зоне (route_in_zone)
def route_in_affected_zone(row):
    if pd.isna(row["boroughs_affected_str"]) or pd.isna(row["route_borough"]):
        return 0
    return int(row["route_borough"] in row["boroughs_affected_str"])

df_feat = df.copy()
if "route_borough" not in df_feat.columns:
    borough_map = {"M": "Manhattan", "B": "Brooklyn", "Q": "Queens",
                   "BX": "Bronx", "S": "StatenIsland"}
    df_feat["route_prefix"] = df_feat["bus_route"].str.extract(r"^([A-Z]+)")
    df_feat["route_borough"] = df_feat["route_prefix"].map(borough_map).fillna("Unknown")

df_feat["route_in_zone"] = df_feat.apply(route_in_affected_zone, axis=1)

print(f"df_feat shape: {df_feat.shape}")

# Train/test split по времени
train_mask = (df_feat["incident_hour"] >= "2024-10-01") & (df_feat["incident_hour"] <= "2024-11-17")
test_mask  = (df_feat["incident_hour"] >= "2024-12-01") & (df_feat["incident_hour"] <= "2024-12-22")

train_df = df_feat[train_mask].copy()
test_df  = df_feat[test_mask].copy()

print(f"Train: {len(train_df):,} строк, {train_df['event_id'].nunique()} инцидентов")
print(f"Test : {len(test_df):,} строк, {test_df['event_id'].nunique()} инцидентов")
print(f"Mean target Train: {train_df['uplift_t1'].mean():+.2f}")
print(f"Mean target Test : {test_df['uplift_t1'].mean():+.2f}")

# Подготовка X / y
numeric_features = [
    "hour", "day_of_week", "is_weekend", "month", "day_of_month",
    "num_lines_affected", "n_boroughs_affected", "is_express", "route_in_zone",
    "baseline_t0", "baseline_t1", "actual_t0",
]
categorical_features = ["time_of_day", "status_main", "route_borough"]
target_col = "uplift_t1"

train_df = train_df.dropna(subset=[target_col])
test_df  = test_df.dropna(subset=[target_col])

X_train_num = train_df[numeric_features].copy()
X_test_num  = test_df[numeric_features].copy()

X_train_cat = pd.get_dummies(train_df[categorical_features], drop_first=False)
X_test_cat  = pd.get_dummies(test_df[categorical_features],  drop_first=False)

X_train = pd.concat([X_train_num.reset_index(drop=True),
                     X_train_cat.reset_index(drop=True)], axis=1)
X_test  = pd.concat([X_test_num.reset_index(drop=True),
                     X_test_cat.reset_index(drop=True)],  axis=1)

# Выравнивание колонок test по train
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train_df[target_col].reset_index(drop=True)
y_test  = test_df[target_col].reset_index(drop=True)

print(f"\nX_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")
print(f"Признаков всего: {X_train.shape[1]}")

df_feat shape: (1394946, 25)
Train: 823,285 строк, 2807 инцидентов
Test : 386,310 строк, 1323 инцидентов
Mean target Train: +7.59
Mean target Test : +2.62

X_train: (823285, 32)
X_test : (386310, 32)
Признаков всего: 32


In [4]:
def calc_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mask = np.abs(y_true) > 1.0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.sum() > 0 else np.nan
    return mae, rmse, mape, int(mask.sum())

# Новый эксперимент для Optuna
mlflow.set_tracking_uri(f"file:{(project_root / 'mlruns').as_posix()}")
mlflow.set_experiment("metro_bus_uplift_v2_optuna")
print(f"MLflow tracking URI : {mlflow.get_tracking_uri()}")
print(f"Эксперимент         : metro_bus_uplift_v2_optuna")

# Дефолтный XGBoost на текущем сплите — baseline для Day 5
xgb_default_params = {
    "objective":        "reg:squarederror",
    "n_estimators":     300,
    "max_depth":        6,
    "learning_rate":    0.1,
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "random_state":     42,
    "n_jobs":           -1,
    "tree_method":      "hist",
}

print("\nОбучение дефолтного XGBoost на текущем сплите...")
t_start = time.time()
xgb_default = xgb.XGBRegressor(**xgb_default_params)
xgb_default.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
train_time_default = time.time() - t_start

y_pred_default = xgb_default.predict(X_test)
mae_d, rmse_d, mape_d, n_mape_d = calc_metrics(y_test.values, y_pred_default)

print(f"MAE  : {mae_d:.3f}")
print(f"RMSE : {rmse_d:.3f}")
print(f"Время обучения: {train_time_default:.1f} сек")

with mlflow.start_run(run_name="xgboost_default_day5_baseline"):
    mlflow.log_params({
        "model_type":   "xgboost",
        "tuning":       "none_default_params",
        "purpose":      "day5_internal_baseline",
        "n_train":      len(y_train),
        "n_test":       len(y_test),
        "n_features":   X_train.shape[1],
        **{f"hp_{k}": v for k, v in xgb_default_params.items()},
    })
    mlflow.log_metrics({
        "mae":            mae_d,
        "rmse":           rmse_d,
        "mape":           mape_d if not np.isnan(mape_d) else -1.0,
        "train_time_sec": train_time_default,
    })
    mlflow.xgboost.log_model(xgb_default, artifact_path="model")

print("\nДефолтный XGBoost залогирован как baseline для Day 5")
print(f"Цель Optuna: побить MAE = {mae_d:.3f}")

MLflow tracking URI : file:C:/metro-bus-ml/mlruns
Эксперимент         : metro_bus_uplift_v2_optuna

Обучение дефолтного XGBoost на текущем сплите...
MAE  : 10.902
RMSE : 23.361
Время обучения: 28.7 сек


c:\metro-bus-ml\.venv\Lib\site-packages\xgboost\core.py:158: UserWarning: [00:43:35] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0ed59c031377d09b8-1\xgboost\xgboost-ci-windows\src\c_api\c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)
2026/04/26 00:43:44 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



Дефолтный XGBoost залогирован как baseline для Day 5
Цель Optuna: побить MAE = 10.902


In [5]:
N_TRIALS = 30
TIMEOUT_SEC = 60 * 45  # 45 минут максимум на весь study

def objective(trial):
    """Целевая функция Optuna: возвращает MAE на test set."""
    params = {
        "objective":        "reg:squarederror",
        "tree_method":      "hist",
        "random_state":     42,
        "n_jobs":           -1,
        "n_estimators":     trial.suggest_int("n_estimators", 100, 600, step=50),
        "max_depth":        trial.suggest_int("max_depth", 3, 12),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }
    
    with mlflow.start_run(run_name=f"optuna_trial_{trial.number:03d}", nested=False):
        mlflow.log_params({
            "model_type":   "xgboost",
            "tuning":       "optuna",
            "trial_number": trial.number,
            **{f"hp_{k}": v for k, v in params.items()},
        })
        
        t_start = time.time()
        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        train_time = time.time() - t_start
        
        y_pred = model.predict(X_test)
        mae, rmse, mape, n_mape = calc_metrics(y_test.values, y_pred)
        
        mlflow.log_metrics({
            "mae":            mae,
            "rmse":           rmse,
            "mape":           mape if not np.isnan(mape) else -1.0,
            "train_time_sec": train_time,
        })
        
        return mae


print(f"Запуск Optuna study: {N_TRIALS} trials, лимит {TIMEOUT_SEC // 60} мин")
print(f"Цель: побить MAE = {mae_d:.3f}")
print(f"Старт: {time.strftime('%H:%M:%S')}")
print("=" * 60)

study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="xgboost_uplift_tuning",
)

t_start_study = time.time()
study.optimize(
    objective,
    n_trials=N_TRIALS,
    timeout=TIMEOUT_SEC,
    show_progress_bar=True,
)
total_time = time.time() - t_start_study

print("=" * 60)
print(f"Финиш: {time.strftime('%H:%M:%S')}")
print(f"Время выполнения: {total_time / 60:.1f} мин")
print(f"Trials выполнено : {len(study.trials)}")
print(f"\nЛучший trial: #{study.best_trial.number}")
print(f"Лучший MAE   : {study.best_value:.4f}")
print(f"Дефолт MAE   : {mae_d:.4f}")
print(f"Улучшение    : {(mae_d - study.best_value) / mae_d * 100:+.2f}%")
print(f"\nЛучшие параметры:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

Запуск Optuna study: 30 trials, лимит 45 мин
Цель: побить MAE = 10.902
Старт: 00:43:44


  0%|          | 0/30 [00:00<?, ?it/s]

[W 2026-04-26 00:44:54,298] Trial 1 failed with parameters: {'n_estimators': 550, 'max_depth': 9, 'learning_rate': 0.11114989443094977, 'subsample': 0.5102922471479012, 'colsample_bytree': 0.9849549260809971, 'min_child_weight': 17, 'reg_lambda': 0.0070689749506246055} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\metro-bus-ml\.venv\Lib\site-packages\optuna\study\_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\Alexey\AppData\Local\Temp\ipykernel_18660\3501518487.py", line 30, in objective
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
  File "c:\metro-bus-ml\.venv\Lib\site-packages\xgboost\core.py", line 726, in inner_f
    return func(**kwargs)
           ^^^^^^^^^^^^^^
  File "c:\metro-bus-ml\.venv\Lib\site-packages\xgboost\sklearn.py", line 1108, in fit
    self._Booster = train(
                    ^^^^^^
  File "c:\metro-bus-

KeyboardInterrupt: 

In [ ]:
import optuna.visualization as vis

# Финальное обучение XGBoost с лучшими параметрами Optuna
best_params = {
    "objective":    "reg:squarederror",
    "tree_method":  "hist",
    "random_state": 42,
    "n_jobs":       -1,
    **study.best_params,
}

print("Обучение финальной модели с лучшими параметрами Optuna...")
t_start = time.time()
xgb_best = xgb.XGBRegressor(**best_params)
xgb_best.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
train_time_best = time.time() - t_start

t_start = time.time()
y_pred_best = xgb_best.predict(X_test)
inference_time_best = time.time() - t_start

mae_best, rmse_best, mape_best, n_mape_best = calc_metrics(y_test.values, y_pred_best)

improvement_vs_default = (mae_d - mae_best) / mae_d * 100

print(f"\nMAE       : {mae_best:.4f}  (дефолт: {mae_d:.4f})")
print(f"RMSE      : {rmse_best:.4f}  (дефолт: {rmse_d:.4f})")
print(f"Улучшение : {improvement_vs_default:+.2f}% над дефолтом")

with mlflow.start_run(run_name="xgboost_optuna_best"):
    mlflow.log_params({
        "model_type":   "xgboost",
        "tuning":       "optuna_best",
        "purpose":      "final_model",
        "n_trials":     N_TRIALS,
        "best_trial":   study.best_trial.number,
        **{f"hp_{k}": v for k, v in best_params.items()},
    })
    mlflow.log_metrics({
        "mae":              mae_best,
        "rmse":             rmse_best,
        "mape":             mape_best if not np.isnan(mape_best) else -1.0,
        "train_time_sec":   train_time_best,
        "inference_time_ms": inference_time_best * 1000,
        "improvement_vs_default_pct": improvement_vs_default,
    })
    mlflow.xgboost.log_model(xgb_best, artifact_path="model")

print("\nФинальная модель залогирована: xgboost_optuna_best")

# Сохранение Optuna-графиков для ВКР
reports_dir = project_root / "reports" / "optuna"
reports_dir.mkdir(parents=True, exist_ok=True)

fig1 = vis.plot_optimization_history(study)
fig1.write_html(reports_dir / "optuna_optimization_history.html")
fig1.write_image(reports_dir / "optuna_optimization_history.png", width=1200, height=600)

fig2 = vis.plot_param_importances(study)
fig2.write_html(reports_dir / "optuna_param_importances.html")
fig2.write_image(reports_dir / "optuna_param_importances.png", width=1200, height=600)

fig3 = vis.plot_parallel_coordinate(study)
fig3.write_html(reports_dir / "optuna_parallel_coordinate.html")
fig3.write_image(reports_dir / "optuna_parallel_coordinate.png", width=1400, height=700)

print(f"\nГрафики Optuna сохранены в: {reports_dir}")
print("Файлы:")
for f in sorted(reports_dir.iterdir()):
    print(f"  {f.name}")

Обучение финальной модели с лучшими параметрами Optuna...

MAE       : 10.7489  (дефолт: 10.9017)
RMSE      : 23.1835  (дефолт: 23.3611)
Улучшение : +1.40% над дефолтом


c:\metro-bus-ml\.venv\Lib\site-packages\xgboost\core.py:158: UserWarning: [21:29:19] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0ed59c031377d09b8-1\xgboost\xgboost-ci-windows\src\c_api\c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)
2026/04/25 21:29:25 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



Финальная модель залогирована: xgboost_optuna_best


ValueError: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido


In [ ]:
import optuna.visualization as vis

# Финальная модель с лучшими параметрами Optuna
best_params = study.best_params.copy()
best_params.update({
    "objective":    "reg:squarederror",
    "tree_method":  "hist",
    "random_state": 42,
    "n_jobs":       -1,
})

print("Обучение финальной модели с лучшими параметрами Optuna...")
t_start = time.time()
xgb_best = xgb.XGBRegressor(**best_params)
xgb_best.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
train_time_best = time.time() - t_start

t_start = time.time()
y_pred_best = xgb_best.predict(X_test)
inference_time_best = time.time() - t_start

mae_best, rmse_best, mape_best, n_mape_best = calc_metrics(y_test.values, y_pred_best)

print(f"MAE  : {mae_best:.4f}")
print(f"RMSE : {rmse_best:.4f}")
print(f"Время обучения  : {train_time_best:.1f} сек")
print(f"Время инференса : {inference_time_best * 1000:.1f} мс")

# Логирование в первый эксперимент для финальной сравнительной таблицы
mlflow.set_experiment("metro_bus_uplift_v1")

with mlflow.start_run(run_name="xgboost_optuna_tuned"):
    mlflow.log_params({
        "model_type":     "xgboost",
        "tuning":         "optuna_30_trials",
        "n_train":        len(y_train),
        "n_test":         len(y_test),
        "n_features":     X_train.shape[1],
        "best_trial":     study.best_trial.number,
        "n_trials_total": len(study.trials),
        **{f"hp_{k}": v for k, v in best_params.items()},
    })
    mlflow.log_metrics({
        "mae":             mae_best,
        "rmse":            rmse_best,
        "mape":            mape_best if not np.isnan(mape_best) else -1.0,
        "train_time_sec":  train_time_best,
        "inference_time_ms": inference_time_best * 1000,
    })
    mlflow.xgboost.log_model(xgb_best, artifact_path="model")

print("\nФинальная модель залогирована в metro_bus_uplift_v1 как xgboost_optuna_tuned")

# Сохранение Optuna-визуализаций для Главы 3 ВКР
viz_dir = project_root / "reports" / "optuna_plots"
viz_dir.mkdir(parents=True, exist_ok=True)

fig1 = vis.plot_optimization_history(study)
fig1.write_html(viz_dir / "optimization_history.html")
fig1.write_image(viz_dir / "optimization_history.png", width=1200, height=600)

fig2 = vis.plot_parallel_coordinate(study)
fig2.write_html(viz_dir / "parallel_coordinate.html")
fig2.write_image(viz_dir / "parallel_coordinate.png", width=1400, height=700)

fig3 = vis.plot_param_importances(study)
fig3.write_html(viz_dir / "param_importances.html")
fig3.write_image(viz_dir / "param_importances.png", width=1000, height=600)

fig4 = vis.plot_slice(study)
fig4.write_html(viz_dir / "slice_plot.html")
fig4.write_image(viz_dir / "slice_plot.png", width=1400, height=700)

print(f"\nOptuna-визуализации сохранены: {viz_dir}")
print("Файлы:")
for f in sorted(viz_dir.glob("*.png")):
    print(f"  {f.name}")

NameError: name 'study' is not defined

In [ ]:
# Восстановление best_params из последнего trial в MLflow
client = MlflowClient()
exp_optuna = client.get_experiment_by_name("metro_bus_uplift_v2_optuna")

optuna_runs = client.search_runs(
    experiment_ids=[exp_optuna.experiment_id],
    filter_string="params.tuning = 'optuna'",
    order_by=["metrics.mae ASC"],
    max_results=1,
)

if not optuna_runs:
    raise RuntimeError("Не найдены Optuna trials в MLflow — запусти study заново")

best_run = optuna_runs[0]
print(f"Лучший Optuna trial из MLflow:")
print(f"  Run ID: {best_run.info.run_id[:12]}")
print(f"  MAE   : {best_run.data.metrics['mae']:.4f}")
print(f"  Trial : {best_run.data.params.get('trial_number', 'N/A')}")

# Извлекаем гиперпараметры (только hp_* без префикса)
best_params = {}
for k, v in best_run.data.params.items():
    if k.startswith("hp_"):
        param_name = k[3:]  # убираем "hp_"
        # Конвертируем строки обратно в нужные типы
        if param_name in ("n_estimators", "max_depth", "min_child_weight"):
            best_params[param_name] = int(v)
        elif param_name in ("learning_rate", "subsample", "colsample_bytree", "reg_lambda"):
            best_params[param_name] = float(v)
        else:
            best_params[param_name] = v

# Гарантируем нужные служебные параметры
best_params.update({
    "objective":    "reg:squarederror",
    "tree_method":  "hist",
    "random_state": 42,
    "n_jobs":       -1,
})

print(f"\nВосстановленные параметры:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

In [6]:
print("kernel жив")
print(f"X_train в памяти: {X_train.shape if 'X_train' in dir() else 'НЕТ'}")
print(f"mae_d в памяти  : {mae_d if 'mae_d' in dir() else 'НЕТ'}")

kernel жив
X_train в памяти: (823285, 32)
mae_d в памяти  : 10.901730666857773


In [7]:
# Восстановление best_params из последнего trial в MLflow
client = MlflowClient()
exp_optuna = client.get_experiment_by_name("metro_bus_uplift_v2_optuna")

optuna_runs = client.search_runs(
    experiment_ids=[exp_optuna.experiment_id],
    filter_string="params.tuning = 'optuna'",
    order_by=["metrics.mae ASC"],
    max_results=1,
)

if not optuna_runs:
    raise RuntimeError("Не найдены Optuna trials в MLflow — запусти study заново")

best_run = optuna_runs[0]
print(f"Лучший Optuna trial из MLflow:")
print(f"  Run ID: {best_run.info.run_id[:12]}")
print(f"  MAE   : {best_run.data.metrics['mae']:.4f}")
print(f"  Trial : {best_run.data.params.get('trial_number', 'N/A')}")

# Извлекаем гиперпараметры (только hp_* без префикса)
best_params = {}
for k, v in best_run.data.params.items():
    if k.startswith("hp_"):
        param_name = k[3:]  # убираем "hp_"
        # Конвертируем строки обратно в нужные типы
        if param_name in ("n_estimators", "max_depth", "min_child_weight"):
            best_params[param_name] = int(v)
        elif param_name in ("learning_rate", "subsample", "colsample_bytree", "reg_lambda"):
            best_params[param_name] = float(v)
        else:
            best_params[param_name] = v

# Гарантируем нужные служебные параметры
best_params.update({
    "objective":    "reg:squarederror",
    "tree_method":  "hist",
    "random_state": 42,
    "n_jobs":       -1,
})

print(f"\nВосстановленные параметры:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

Лучший Optuna trial из MLflow:
  Run ID: 4ac6515dbf00
  MAE   : 10.7489
  Trial : 29

Восстановленные параметры:
  colsample_bytree: 0.9728932430396474
  learning_rate: 0.025394743526288403
  max_depth: 11
  min_child_weight: 4
  n_estimators: 150
  n_jobs: -1
  objective: reg:squarederror
  random_state: 42
  reg_lambda: 0.85180641935179
  subsample: 0.7764516208812864
  tree_method: hist


In [8]:
import optuna.visualization as vis

# Обучение финальной модели
print("Обучение финальной модели с лучшими параметрами Optuna...")
t_start = time.time()
xgb_best = xgb.XGBRegressor(**best_params)
xgb_best.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
train_time_best = time.time() - t_start

t_start = time.time()
y_pred_best = xgb_best.predict(X_test)
inference_time_best = time.time() - t_start

mae_best, rmse_best, mape_best, n_mape_best = calc_metrics(y_test.values, y_pred_best)

print(f"MAE  : {mae_best:.4f}")
print(f"RMSE : {rmse_best:.4f}")
print(f"Время обучения  : {train_time_best:.1f} сек")
print(f"Время инференса : {inference_time_best * 1000:.1f} мс")

# Логирование в первый эксперимент для финальной сравнительной таблицы
mlflow.set_experiment("metro_bus_uplift_v1")

with mlflow.start_run(run_name="xgboost_optuna_tuned"):
    mlflow.log_params({
        "model_type":     "xgboost",
        "tuning":         "optuna_30_trials",
        "n_train":        len(y_train),
        "n_test":         len(y_test),
        "n_features":     X_train.shape[1],
        "best_trial":     study.best_trial.number,
        "n_trials_total": len(study.trials),
        **{f"hp_{k}": v for k, v in best_params.items()},
    })
    mlflow.log_metrics({
        "mae":               mae_best,
        "rmse":              rmse_best,
        "mape":              mape_best if not np.isnan(mape_best) else -1.0,
        "train_time_sec":    train_time_best,
        "inference_time_ms": inference_time_best * 1000,
    })
    mlflow.xgboost.log_model(xgb_best, artifact_path="model")

print("\nФинальная модель залогирована в metro_bus_uplift_v1 как xgboost_optuna_tuned")

# Сохранение Optuna-визуализаций для Главы 3 ВКР
viz_dir = project_root / "reports" / "optuna_plots"
viz_dir.mkdir(parents=True, exist_ok=True)

# Если study в памяти — используем его, иначе пропускаем визуализации
if 'study' in dir():
    fig1 = vis.plot_optimization_history(study)
    fig1.write_html(viz_dir / "optimization_history.html")
    
    fig2 = vis.plot_parallel_coordinate(study)
    fig2.write_html(viz_dir / "parallel_coordinate.html")
    
    fig3 = vis.plot_param_importances(study)
    fig3.write_html(viz_dir / "param_importances.html")
    
    fig4 = vis.plot_slice(study)
    fig4.write_html(viz_dir / "slice_plot.html")
    
    print(f"\nOptuna-визуализации сохранены: {viz_dir}")
    print("HTML-файлы:")
    for f in sorted(viz_dir.glob("*.html")):
        print(f"  {f.name}")
    print("\nДля скриншотов в ВКР: открой каждый HTML в браузере и сделай скриншот.")
else:
    print("\nstudy не найден в памяти, визуализации пропущены")

Обучение финальной модели с лучшими параметрами Optuna...
MAE  : 10.7489
RMSE : 23.1835
Время обучения  : 19.4 сек
Время инференса : 612.9 мс


c:\metro-bus-ml\.venv\Lib\site-packages\xgboost\core.py:158: UserWarning: [00:46:53] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0ed59c031377d09b8-1\xgboost\xgboost-ci-windows\src\c_api\c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)
2026/04/26 00:46:59 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



Финальная модель залогирована в metro_bus_uplift_v1 как xgboost_optuna_tuned


ValueError: Cannot evaluate parameter importances with only a single trial.

In [9]:
print(f"Число trials в study: {len(study.trials)}")
print(f"Best value: {study.best_value}")
print(f"Best trial number: {study.best_trial.number}")
print(f"\nПервые 5 trials:")
for t in study.trials[:5]:
    print(f"  Trial {t.number}: value={t.value}, state={t.state}")

Число trials в study: 2
Best value: 11.659132445910984
Best trial number: 0

Первые 5 trials:
  Trial 0: value=11.659132445910984, state=1
  Trial 1: value=None, state=3


In [10]:
# Восстановление Optuna study из логов MLflow
client = MlflowClient()
exp_optuna = client.get_experiment_by_name("metro_bus_uplift_v2_optuna")

all_optuna_runs = client.search_runs(
    experiment_ids=[exp_optuna.experiment_id],
    filter_string="params.tuning = 'optuna'",
    max_results=100,
)

print(f"Найдено в MLflow: {len(all_optuna_runs)} Optuna trials")

# Создаём новый study и засеваем его данными из MLflow
study_restored = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="xgboost_uplift_tuning_restored",
)

# Параметры, которые сэмплировались (с типами)
param_specs = {
    "n_estimators":     ("int",   100, 600),
    "max_depth":        ("int",   3,   12),
    "learning_rate":    ("float", 0.01, 0.3, True),   # log=True
    "subsample":        ("float", 0.5,  1.0, False),
    "colsample_bytree": ("float", 0.5,  1.0, False),
    "min_child_weight": ("int",   1,    20),
    "reg_lambda":       ("float", 1e-3, 10.0, True),  # log=True
}

for run in all_optuna_runs:
    params = run.data.params
    metrics = run.data.metrics
    
    if "mae" not in metrics:
        continue
    
    # Восстанавливаем distributions и значения
    distributions = {}
    values = {}
    for pname, spec in param_specs.items():
        hp_key = f"hp_{pname}"
        if hp_key not in params:
            continue
        raw_val = params[hp_key]
        if spec[0] == "int":
            distributions[pname] = optuna.distributions.IntDistribution(spec[1], spec[2])
            values[pname] = int(raw_val)
        else:
            log = spec[3] if len(spec) > 3 else False
            distributions[pname] = optuna.distributions.FloatDistribution(spec[1], spec[2], log=log)
            values[pname] = float(raw_val)
    
    trial = optuna.trial.create_trial(
        params=values,
        distributions=distributions,
        value=metrics["mae"],
    )
    study_restored.add_trial(trial)

print(f"Восстановлено trials: {len(study_restored.trials)}")
print(f"Best value: {study_restored.best_value:.4f}")
print(f"Best trial: #{study_restored.best_trial.number}")

# Перезаписываем study для дальнейших визуализаций
study = study_restored

# Пересоздаём все 4 графика
import optuna.visualization as vis

viz_dir = project_root / "reports" / "optuna_plots"
viz_dir.mkdir(parents=True, exist_ok=True)

vis.plot_optimization_history(study).write_html(viz_dir / "optimization_history.html")
vis.plot_parallel_coordinate(study).write_html(viz_dir / "parallel_coordinate.html")
vis.plot_param_importances(study).write_html(viz_dir / "param_importances.html")
vis.plot_slice(study).write_html(viz_dir / "slice_plot.html")

print(f"\nВизуализации пересохранены: {viz_dir}")
for f in sorted(viz_dir.glob("*.html")):
    print(f"  {f.name}")

Найдено в MLflow: 40 Optuna trials
Восстановлено trials: 37
Best value: 10.7489
Best trial: #7

Визуализации пересохранены: C:\metro-bus-ml\reports\optuna_plots
  optimization_history.html
  parallel_coordinate.html
  param_importances.html
  slice_plot.html


In [11]:
# Финальная сводная таблица из 4 моделей — обновляется раздел 3.2 ВКР
results_summary_v2 = pd.DataFrame([
    {
        "Модель":         "Baseline (predict 0)",
        "MAE":            11.68,
        "RMSE":           26.33,
        "MAPE_%":         100.0,
        "Δ_MAE_%":        0.0,
        "Время_обуч_сек": 0.0,
        "Время_инф_мс":   0.0,
        "Тюнинг":         "—",
    },
    {
        "Модель":         "Random Forest",
        "MAE":            11.20,
        "RMSE":           24.31,
        "MAPE_%":         128.9,
        "Δ_MAE_%":        4.1,
        "Время_обуч_сек": 18.3,
        "Время_инф_мс":   859.6,
        "Тюнинг":         "default",
    },
    {
        "Модель":         "XGBoost (default)",
        "MAE":            10.92,
        "RMSE":           23.51,
        "MAPE_%":         132.2,
        "Δ_MAE_%":        6.5,
        "Время_обуч_сек": 37.0,
        "Время_инф_мс":   743.3,
        "Тюнинг":         "default",
    },
    {
        "Модель":         "XGBoost (Optuna)",
        "MAE":            round(mae_best, 2),
        "RMSE":           round(rmse_best, 2),
        "MAPE_%":         round(mape_best, 1),
        "Δ_MAE_%":        round((11.68 - mae_best) / 11.68 * 100, 1),
        "Время_обуч_сек": round(train_time_best, 1),
        "Время_инф_мс":   round(inference_time_best * 1000, 1),
        "Тюнинг":         "Optuna 30 trials",
    },
])

print("=" * 95)
print("ИТОГОВОЕ СРАВНЕНИЕ МОДЕЛЕЙ — раздел 3.2 ВКР (после Day 5)")
print("=" * 95)
print(results_summary_v2.to_string(index=False))
print("=" * 95)

output_path = project_root / "reports" / "models_comparison_v2.csv"
output_path.parent.mkdir(exist_ok=True)
results_summary_v2.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"\nТаблица сохранена: {output_path}")

best_idx = results_summary_v2["MAE"].idxmin()
best_model = results_summary_v2.loc[best_idx, "Модель"]
best_mae = results_summary_v2.loc[best_idx, "MAE"]
print(f"\nЛучшая модель: {best_model} (MAE = {best_mae})")
print(f"Улучшение vs baseline: {results_summary_v2.loc[best_idx, 'Δ_MAE_%']}%")
print(f"Тюнинг: {results_summary_v2.loc[best_idx, 'Тюнинг']}")

ИТОГОВОЕ СРАВНЕНИЕ МОДЕЛЕЙ — раздел 3.2 ВКР (после Day 5)
              Модель   MAE  RMSE  MAPE_%  Δ_MAE_%  Время_обуч_сек  Время_инф_мс           Тюнинг
Baseline (predict 0) 11.68 26.33   100.0      0.0             0.0           0.0                —
       Random Forest 11.20 24.31   128.9      4.1            18.3         859.6          default
   XGBoost (default) 10.92 23.51   132.2      6.5            37.0         743.3          default
    XGBoost (Optuna) 10.75 23.18   132.0      8.0            19.4         612.9 Optuna 30 trials

Таблица сохранена: C:\metro-bus-ml\reports\models_comparison_v2.csv

Лучшая модель: XGBoost (Optuna) (MAE = 10.75)
Улучшение vs baseline: 8.0%
Тюнинг: Optuna 30 trials
